# ⭐ Project · The Nimbus Briefing Assistant

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama and this section's
> `mcp_server.py`. The full run makes a few dozen model calls; on a local model
> expect several minutes.

**Nimbus Robotics** (the fictional company in `data/kb/`) is rolling out 180
delivery robots to its customer Albers Markt, and the rollout is slipping. Albers
Markt asked for *"a written briefing with a realistic date per store group"*. The
facts are spread across six documents (a battery shortage, a delayed software
release, a permit, customer feedback), so no single lookup answers it.

You'll build the assistant as a **CrewAI Flow**: the Flow guarantees the
structure, and small crews do the judgement work inside each step. It's the same
project as in the other framework sections, so you can compare the designs.

**What you'll practise**

| Feature | From notebook |
|---|---|
| A Flow with typed state, routers and a guarded loop | 06 |
| Crews inside flow steps, each with a clear role | 01–02, 05 |
| MCP tools via `MCPServerAdapter`, filtered per crew | 11 |
| Parallel research with `kickoff_for_each_async` | 05, 12 |
| Guardrails: leaked tool calls, and exact checks in code | 07, 11 |
| `@persist` + idempotent steps: surviving a crash | 09 |
| `@human_feedback` approval before saving, paused and resumed | 08 |
| Verifying the side effect, and token usage per step | 06, 02 |

## 1 · The design

```
 @start plan ──► research ──► write ◄──────── 'revise' ─┐
 (planner crew,   (one crew per question,   (writer   review ──(router)──┤
  output_pydantic) in parallel, read-only    crew)    (code checks +     └─ 'ready' ─► request_approval
                   MCP tools + leak guard)             reviewer crew)                  (@human_feedback: pauses)
                                                                                       ├─ 'approved' ─► save (MCP) ─► verify
                                                                                       └─ 'rejected' ─► stop
 @persist: state saved after every step → a restarted flow skips steps whose results are already in state
```

Design decisions, and why:

- **The Flow owns the structure** (notebook 06); crews only do the work inside steps.
- **Every step is idempotent** (notebook 09): `@persist` restores state but
  restarts from `@start`, so each step first checks whether its result already exists.
- **Least privilege** (notebook 11): research crews get read-only MCP tools; only the
  `save` step touches `save_report`, and only after approval.
- **Don't trust, verify**: exact checks in code, the leaked-tool-call guardrail, and a
  file-system check after saving.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import asyncio, json, pathlib, sys, shutil
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew
from crewai.flow import Flow, start, listen, router, or_, persist, human_feedback, HumanFeedbackPending
from crewai.flow.async_feedback import HumanFeedbackProvider
from crewai.flow.persistence import SQLiteFlowPersistence
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters
from common import SECTION_DIR, reject_leaked_tool_calls

SERVER = StdioServerParameters(command=sys.executable, args=[str(SECTION_DIR / 'mcp_server.py')])
REPORTS = SECTION_DIR / 'data' / 'reports'
WORK = pathlib.Path('checkpoints/project')
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

with MCPServerAdapter(SERVER) as tools:                      # a quick check that the server works
    print('MCP tools available:', [t.name for t in tools])

## 2 · State, and the small crews

The state holds everything a restarted flow needs, including counters that make
the loop and the idempotency checks possible. Each crew is built by a small
function with a **fresh LLM**, so token usage is measured per step.

In [ ]:
DOC_NAMES = ['01_company_overview', '02_q3_rollout_plan', '03_incident_report_battery',
             '04_software_release_4_2', '05_permits_and_regulation', '06_customer_feedback_albers']
REQUIRED_SECTIONS = ('## Summary', '## Go-live dates', '## Risks', '## Sources')
MAX_REVISIONS = 2


class BriefingState(BaseModel):
    request: str = ''
    questions: list[str] = []
    findings: list[str] = []
    draft: str = ''
    revision: int = 0                 # drafts written so far
    reviewed_revision: int = 0        # last draft that was reviewed
    feedback: str = ''
    review_passed: bool = False
    report_path: str = ''
    usage: dict[str, int] = {}


class Plan(BaseModel):
    questions: list[str] = Field(description='2 to 4 specific research questions')

class Verdict(BaseModel):
    supported: bool
    problems: str


def planner_crew():
    a = Agent(role='Research planner', goal='Plan research for customer briefings',
              backstory='You write 2-4 specific research questions. Each question is a separate item.', llm=get_llm())
    return Crew(agents=[a], tasks=[Task(description='Request: {request}\nAvailable documents: {docs}',
                                        expected_output='2-4 research questions', agent=a, output_pydantic=Plan)])

def research_crew(kb_tools):
    a = Agent(role='Researcher', goal='Answer one question from the Nimbus knowledge base',
              backstory='You search, then READ the relevant documents, then answer in at most 4 sentences with '
                        'concrete numbers and dates. You end with "Sources: <document names>". You never guess.',
              llm=get_llm(), tools=list(kb_tools))
    return Crew(agents=[a], tasks=[Task(description='Context: {request}\nQuestion: {question}',
                                        expected_output='A short factual answer with sources', agent=a,
                                        guardrail=reject_leaked_tool_calls, guardrail_max_retries=2)])

def writer_crew():
    a = Agent(role='Briefing writer', goal='Write accurate customer briefings',
              backstory='You use ONLY the facts given and never invent numbers or dates.', llm=get_llm())
    return Crew(agents=[a], tasks=[Task(
        description='Write a briefing for Albers Markt in markdown with exactly these sections: ## Summary, '
                    '## Go-live dates per store group, ## Risks, ## Sources. In Sources list the knowledge-base '
                    'document names used.\n\nFindings:\n{findings}\n\n{feedback}',
        expected_output='The briefing in markdown', agent=a)])

def reviewer_crew():
    a = Agent(role='Fact checker', goal='Catch unsupported claims', backstory='Strict and specific.', llm=get_llm())
    return Crew(agents=[a], tasks=[Task(
        description='Findings:\n{findings}\n\nBriefing:\n{draft}\n\nIs every claim in the briefing supported by '
                    'the findings? List unsupported or wrong claims, or write "none".',
        expected_output='A verdict', agent=a, output_pydantic=Verdict)])

## 3 · The Flow

Read it top to bottom; it *is* the design diagram. Every step starts with an
idempotency check. `FAIL_WRITE_ONCE` lets us simulate a crash in §4.

In [ ]:
FAIL_WRITE_ONCE = {'armed': False}
STORE = SQLiteFlowPersistence(str(WORK / 'briefing_flows.db'))


class PauseForReviewer(HumanFeedbackProvider):
    def request_feedback(self, context, flow):
        raise HumanFeedbackPending(context=context)          # save and return; resume later


def add_usage(state, step, out):
    state.usage[step] = state.usage.get(step, 0) + out.token_usage.total_tokens


@persist(STORE)
class BriefingFlow(Flow[BriefingState]):

    @start()
    async def plan(self):
        if self.state.questions:
            print('  ↺ plan (restored)'); return
        out = await planner_crew().kickoff_async(inputs={'request': self.state.request, 'docs': ', '.join(DOC_NAMES)})
        add_usage(self.state, 'plan', out)
        self.state.questions = [q for q in out.pydantic.questions if q.strip()][:4] or [self.state.request]
        print('  ▶ plan:', *[f'\n      • {q[:100]}' for q in self.state.questions])

    @listen(plan)
    async def research(self):
        if self.state.findings:
            print('  ↺ research (restored)'); return
        with MCPServerAdapter(SERVER, 'list_documents', 'search_documents', 'read_document') as kb_tools:
            crew = research_crew(kb_tools)
            outs = await crew.kickoff_for_each_async(
                inputs=[{'request': self.state.request, 'question': q} for q in self.state.questions])
        for o in outs:
            add_usage(self.state, 'research', o)
        self.state.findings = [f'### {q}\n{o.raw}' for q, o in zip(self.state.questions, outs)]
        if not self.state.findings:
            raise RuntimeError('research produced no findings; refusing to write from nothing')
        print(f'  ▶ research: {len(self.state.findings)} findings')

    @listen(or_(research, 'revise'))
    async def write(self):
        if self.state.revision > self.state.reviewed_revision:
            print('  ↺ write (restored draft awaiting review)'); return
        if FAIL_WRITE_ONCE['armed']:
            FAIL_WRITE_ONCE['armed'] = False
            raise RuntimeError('simulated crash: the writer died mid-run')
        feedback = f'Fix these problems from the previous draft:\n{self.state.feedback}' if self.state.feedback else ''
        out = await writer_crew().kickoff_async(inputs={'findings': '\n\n'.join(self.state.findings),
                                                        'feedback': feedback})
        add_usage(self.state, 'write', out)
        self.state.draft, self.state.revision = out.raw, self.state.revision + 1
        print(f'  ▶ write: draft v{self.state.revision}')

    @listen(write)
    async def review(self):
        if self.state.reviewed_revision == self.state.revision:
            return
        problems = [f'missing section "{s}"' for s in REQUIRED_SECTIONS if s.lower() not in self.state.draft.lower()]
        if not any(d in self.state.draft for d in DOC_NAMES):
            problems.append('Sources must name knowledge-base documents such as 03_incident_report_battery')
        if not problems:                                            # exact checks first, then judgement
            out = await reviewer_crew().kickoff_async(inputs={'findings': '\n\n'.join(self.state.findings),
                                                              'draft': self.state.draft})
            add_usage(self.state, 'review', out)
            if not out.pydantic.supported:
                problems.append(out.pydantic.problems)
        self.state.feedback = '; '.join(problems)
        self.state.review_passed = not problems
        self.state.reviewed_revision = self.state.revision
        print(f'  ▶ review v{self.state.revision}:', 'PASS' if not problems else self.state.feedback[:140])

    @router(review)
    def decide(self):
        if self.state.review_passed or self.state.revision >= MAX_REVISIONS:
            return 'ready'
        return 'revise'

    @listen('ready')
    @human_feedback(message='Approve this briefing for saving?', emit=['approved', 'rejected'],
                    llm=get_llm(), provider=PauseForReviewer())
    def request_approval(self):
        status = 'passed review' if self.state.review_passed else f'FLAGGED: {self.state.feedback}'
        return f'[{status}]\n\n{self.state.draft}'

    @listen('approved')
    def save(self):
        if self.state.report_path:
            return self.state.report_path
        with MCPServerAdapter(SERVER, 'save_report') as tools:
            self.state.report_path = str(tools[0].run(title='Albers Markt rollout briefing',
                                                      markdown=self.state.draft)).strip()
        return self.state.report_path

    @listen('rejected')
    def stop(self):
        return 'rejected by the reviewer: nothing saved'

## 4 · Run it, with a crash in the middle

We arm `FAIL_WRITE_ONCE` so the writer fails the first time, after planning and
research have finished and been **persisted**. Then we kick off again with the
same flow `id`, as a restarted service would. Watch which steps say "restored".

In [ ]:
REQUEST = ('Albers Markt asked for a written briefing on the Courier C2 rollout to their 12 stores, '
           'with a realistic go-live date per store group and the main risks.')

FAIL_WRITE_ONCE['armed'] = True
flow = BriefingFlow(suppress_flow_events=True)
print('attempt 1:')
try:
    await flow.kickoff_async(inputs={'request': REQUEST})
except Exception as err:
    print('\n💥', err)
FLOW_ID = flow.state.id
print('flow id:', FLOW_ID)

In [ ]:
print('attempt 2 (a restarted process would do exactly this):')
flow = BriefingFlow(suppress_flow_events=True)
result = await flow.kickoff_async(inputs={'id': FLOW_ID})
print('\nkickoff returned:', type(result).__name__, '← paused for the human reviewer')

Planning and research were **restored** from the persisted state, so no planner or
research crews ran again. The writer, which had crashed, ran, then the review
loop, and the flow **paused** at `request_approval`, waiting for a human.

## 5 · The human review

In [ ]:
pending = BriefingFlow.from_pending(FLOW_ID, STORE, suppress_flow_events=True)
print(pending.pending_feedback.method_output)

Read the draft critically, as the human in the loop. Are the dates consistent
with the documents? The battery incident, the 4.2 release estimate and the
customer's "8 of 12 by 30 September" condition all matter.

**What our test run showed.** The draft reached the human **flagged** after two
revisions, and the flags themselves were partly confused: the fact checker
complained that the briefing claimed "180 robots by 30 September" when the
findings said exactly that. Meanwhile it missed a real error: the draft called
the **Utrecht permit** the main risk, although `05_permits_and_regulation` says the
permit was granted on 8 August. A model reviewer is another model; it both
over-flags and under-flags. That's why a human decides, and why the exercises
add stricter code checks. Your run will differ, so read your draft the same way.

Then approve, or reject with a reason. We approve here; try a rejection in your own run.

In [ ]:
before = set(REPORTS.glob('*.md')) if REPORTS.exists() else set()
final = await pending.resume_async('Approved. Please save it.')
new_files = sorted(set(REPORTS.glob('*.md')) - before)

print('outcome         :', pending.last_human_feedback.outcome)
print('flow says saved :', final)
print('verified on disk:', [f.relative_to(SECTION_DIR).as_posix() for f in new_files] or 'NOTHING SAVED')
print('tokens per step :', pending.state.usage, '| total', sum(pending.state.usage.values()))

Notice where the tokens went: **research dominated** (about 47,000 of 55,000
tokens in our run), because each research crew read full documents, and a
retried task reads them again. That's the first place to optimise: smaller
documents, `search_documents` snippets instead of full reads, or fewer questions.

## 6 · Compare with the other sections

| Design decision | LangGraph | OpenAI Agents SDK | CrewAI |
|---|---|---|---|
| The fixed flow | a graph | an async Python function | a **Flow** class with `@start`/`@listen`/`@router` |
| Parallel research | `Send` + reducer | `asyncio.gather` | `kickoff_for_each_async` |
| Workers | `create_agent` in a node | `Agent` with MCP tools | a small **crew** per question |
| Crash recovery | automatic checkpoints, resume at the next step | our own `step()` helper | `@persist`, restart from `@start` + **idempotent steps** |
| Human approval | `interrupt()` | approval-gated tool + `RunState` | `@human_feedback` + non-blocking provider |
| Refine loop | counter in state + router | a `for` loop | counter in state + `@router` + `or_(…, 'revise')` |

## 7 · Exercises

1. **Stricter review** (07): add a code check that every date in the draft appears in some finding.
2. **Reject path** (08): on `rejected`, route back to `write` with the reviewer's words as feedback.
3. **Memory** (09): give the writer crew a `Memory` so the customer's preferences carry over to the next briefing.
4. **Metrics** (10): add an event listener that counts tool calls per research question.
5. **Tests** (12): build `BriefingFlow` with a scripted `BaseLLM` and test that a failing review loops at most `MAX_REVISIONS` times.

## 8 · Check yourself

<details><summary><b>1.</b> Why does every step start with an "already done?" check?</summary>

`@persist` restores the state but restarts the flow from `@start`. Without the checks, a restored flow would re-plan, re-research and re-write. The checks make each step idempotent.
</details>

<details><summary><b>2.</b> Why do the research crews get only read tools, and where does <code>save_report</code> get called?</summary>

Least privilege: research never needs to write. `save_report` is called only in the `save` step, which runs only after the reviewer approved.
</details>

<details><summary><b>3.</b> What does the <code>reject_leaked_tool_calls</code> guardrail protect against here?</summary>

A local model writing a tool call as text; that text would otherwise become a "finding", and the writer would build the briefing on it (notebook 11).
</details>

<details><summary><b>4.</b> How does this design's crash recovery differ from LangGraph's?</summary>

LangGraph resumes at the next step from an exact checkpoint. The CrewAI flow restores its state and re-runs from `@start`, relying on idempotent steps to skip finished work.
</details>

## Section complete 🎉

You've built a planned, parallel, self-reviewing, crash-tolerant, human-approved
research workflow with CrewAI: a **Flow** for the structure, and small **crews**
for the work inside it.

**Where to go next:** build the same project in another framework section, then
compare the versions side by side in the comparison section.